# Notebook 22 — MJO NSV Stage 2: Intrinsic Dimension Estimation  *(CRITICAL RESULT)*
**Project:** ENSO-BSISO SSL — MJO NSV extension  
**Author:** Jiayi (jh9141@nyu.edu)

**Stage 2 of MJO NSV pipeline.** nb21 produced 64-D Stage 1 latents that manifoldized cleanly (PC1=76.0%, +63.4% over persistence). This notebook estimates the intrinsic dimension `d̂` of that latent point cloud.

## Three-hypothesis decision tree (Session 34)

| | d̂ | Interpretation |
|---|:-:|---|
| **H1** | 2 | RMM is dimension-sufficient. **Structural asymmetry with BSISO** (where d̂=4): MJO is a single equatorial coupled mode, BSISO has multiple monsoon-Rossby axes. |
| **H2** | 3 | The 3rd axis is amplitude or ENSO. Less than BSISO's 4 but still beyond RMM convention. |
| **H3** | ≥ 4 | Matches BSISO. Both intraseasonal modes have undercounted state spaces — strongest possible result. |

## Method (same as BSISO nb19, MJO-specific paths and labels)

- **Levina-Bickel MLE** with k-sweep: `k = int(N × {0.008, 0.010, 0.012, 0.014, 0.016})` — Chen et al. recipe.
- **Two-NN** (Facco et al. 2017) and **local PCA** as cross-checks.
- **Sanity controls**: Gaussian noise (expect d̂ ≈ ambient saturation ceiling ≈ 35–40 at our N), shuffled-z (expect d̂ ≫ real if manifold is genuine).
- **ENSO stratification**: ID estimated separately on EN/Neutral/LN subsets. H1 → all 3 equal global; H2 → within-cat ≈ global−1 (ENSO removed an axis).
- **N-aware confidence threshold** (Session 32 patch): trust d̂ if noise control gives ≥ 3× headroom above real d̂.

## Inputs

- `MJO/nsv/latents_lag10/z_train.npy` shape `(12969, 64)` ← from nb21
- `MJO/nsv/latents_lag10/z_val.npy` shape `(3107, 64)`
- `MJO/nsv/latents_lag10/{rmm_phase_t, rmm_amplitude_t, enso_cat_t, weak_mjo_t, train_mask}.npy` (label arrays nb21 copied here)

Larger N than BSISO (16,076 unique vs 3,999) → Levina-Bickel reliable up to **d̂ ≈ 14**.

## Outputs (`MJO/nsv/results/stage2_lag10/`)

- `intrinsic_dim.json`
- `id_lb_sweep.png`, `controls.png`, `id_by_enso.png`, `pca_visualization.png`
- `stage2_summary.md`

## Decision rule for nb23

After this notebook: integer-rounded `d̂` sets the SIREN bottleneck in nb23 (Stage 3).

## Runtime ~3–5 min on Colab T4 (skdim is fast; larger N than BSISO means slightly more wall time).

---

## Cell 1 — Setup: Install skdim, Load MJO Latents + Labels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-dimension==0.3.4

import os, json
import numpy as np
import matplotlib.pyplot as plt
import skdim

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
NSV_DIR     = f'{MJO_DIR}/nsv'
LATENT_DIR  = f'{NSV_DIR}/latents_lag10'
RESULTS_DIR = f'{NSV_DIR}/results/stage2_lag10'
os.makedirs(RESULTS_DIR, exist_ok=True)

for fn in ['z_train.npy', 'z_val.npy', 'train_mask.npy', 'rmm_phase_t.npy', 'enso_cat_t.npy']:
    p = f'{LATENT_DIR}/{fn}'
    assert os.path.exists(p), f'Missing: {p} — did nb21 run?'

SEED = 42
rng = np.random.default_rng(SEED)

# Load latents and labels (nb21 saved labels in latents_lag10/, so this is one-stop)
z_train    = np.load(f'{LATENT_DIR}/z_train.npy')
z_val      = np.load(f'{LATENT_DIR}/z_val.npy')
z_all      = np.concatenate([z_train, z_val], axis=0)
train_mask = np.load(f'{LATENT_DIR}/train_mask.npy')
phase_t    = np.load(f'{LATENT_DIR}/rmm_phase_t.npy')
enso_t     = np.load(f'{LATENT_DIR}/enso_cat_t.npy')

# Re-order labels to match z_all ordering [train..., val...]
order = np.concatenate([np.where(train_mask)[0], np.where(~train_mask)[0]])
phase_all = phase_t[order]
enso_all  = enso_t[order]

# Deduplicate (skdim requires this — duplicate pts make NN distances zero)
_, unique_idx = np.unique(z_all, axis=0, return_index=True)
unique_idx = np.sort(unique_idx)
z = z_all[unique_idx]
phase = phase_all[unique_idx]
enso  = enso_all[unique_idx]
N, D = z.shape

print(f'Loaded MJO latents:  z_train {z_train.shape}, z_val {z_val.shape}, combined unique {z.shape}')
print(f'Latent stats:        mean ||z|| = {np.linalg.norm(z, axis=1).mean():.4f}')
print(f'                     per-dim std min = {z.std(0).min():.4f}, max = {z.std(0).max():.4f}')
assert N > 500, f'Too few unique points ({N}) for stable ID estimation'
print(f'✓ Deduplication kept {N}/{z_all.shape[0]} ({100*N/z_all.shape[0]:.1f}%) unique points.')
print(f'\nlog₂(N) ≈ {np.log2(N):.1f}  →  Levina-Bickel reliable up to d̂ ≈ {int(np.log2(N))}.')

z_centered = z - z.mean(axis=0)

## Cell 2 — Levina-Bickel MLE with k-sweep

In [ ]:
k_fractions = [0.008, 0.010, 0.012, 0.014, 0.016]
k_list = [max(int(N * c), 3) for c in k_fractions]
print(f'N = {N}  →  k_list = {k_list}')

id_lb_by_k = []
for k in k_list:
    est = skdim.id.MLE(K=k); est.fit(z)
    id_lb_by_k.append(float(est.dimension_))
    print(f'  k = {k:4d}  →  ID_LB = {id_lb_by_k[-1]:.3f}')

id_lb_by_k = np.asarray(id_lb_by_k)
LB_mean = float(id_lb_by_k.mean())
LB_std  = float(id_lb_by_k.std())
print(f'\nLevina-Bickel:  d̂ = {LB_mean:.3f} ± {LB_std:.3f}  (across {len(k_list)} k values)')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_list, id_lb_by_k, 'o-', color='steelblue', lw=2, ms=10, label='LB per k')
ax.axhline(LB_mean, color='red', ls='--', lw=1.5, label=f'Mean = {LB_mean:.2f}')
ax.fill_between(k_list, LB_mean - LB_std, LB_mean + LB_std, color='red', alpha=0.15, label=f'±1σ = ±{LB_std:.2f}')
ax.set_xlabel('Neighborhood size k'); ax.set_ylabel('ID estimate')
ax.set_title(f'MJO Levina-Bickel ID sweep over k  (N = {N})', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/id_lb_sweep.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 3 — Cross-Check Estimators + Sanity Controls

In [ ]:
est_tnn = skdim.id.TwoNN(); est_tnn.fit(z)
id_tnn = float(est_tnn.dimension_)
print(f'Two-NN:                {id_tnn:.3f}')

est_lpca = skdim.id.lPCA(); est_lpca.fit(z)
id_lpca = float(est_lpca.dimension_)
print(f'local PCA:             {id_lpca:.3f}')

# Control A: Gaussian noise (calibration)
z_noise = rng.standard_normal((N, D)).astype(np.float32)
id_noise_tnn = float(skdim.id.TwoNN().fit(z_noise).dimension_)
id_noise_mle = float(skdim.id.MLE(K=k_list[2]).fit(z_noise).dimension_)
print(f'\n[Control A] Gaussian noise in R^{D}, N={N}:')
print(f'    Two-NN = {id_noise_tnn:.2f}   MLE = {id_noise_mle:.2f}   (saturation ceiling at this N)')

# Control B: per-dim shuffled z
z_shuffled = z.copy()
for d_i in range(D):
    rng.shuffle(z_shuffled[:, d_i])
id_shuf_tnn = float(skdim.id.TwoNN().fit(z_shuffled).dimension_)
id_shuf_mle = float(skdim.id.MLE(K=k_list[2]).fit(z_shuffled).dimension_)
print(f'\n[Control B] Per-dim shuffled z (real marginals, destroyed correlations):')
print(f'    Two-NN = {id_shuf_tnn:.2f}   MLE = {id_shuf_mle:.2f}   (should be > {LB_mean:.1f} if manifold is real)')

print('\n' + '=' * 70)
print(f'MJO INTRINSIC DIMENSION SUMMARY  (N={N}, ambient D={D})')
print('=' * 70)
print(f'  Levina-Bickel (k-sweep)    : {LB_mean:.3f} ± {LB_std:.3f}')
print(f'  Two-NN                     : {id_tnn:.3f}')
print(f'  local PCA                  : {id_lpca:.3f}')
print(f'  ── controls ──')
print(f'  Gaussian noise (Two-NN)    : {id_noise_tnn:.3f}   (saturation ceiling)')
print(f'  Shuffled z (Two-NN)        : {id_shuf_tnn:.3f}   (real data must be lower)')

fig, ax = plt.subplots(figsize=(11, 5))
labels = ['LB (k-sweep)', 'Two-NN', 'lPCA', 'Gaussian noise\n(Two-NN)', 'Shuffled z\n(Two-NN)']
vals   = [LB_mean,         id_tnn,   id_lpca, id_noise_tnn,                id_shuf_tnn]
errs   = [LB_std,          0,        0,       0,                           0]
colors = ['steelblue',     '#1f77b4','#1f77b4', '#888888',                  '#d62728']
bars = ax.bar(labels, vals, yerr=errs, color=colors, alpha=0.85, capsize=6)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(D, color='gray', ls=':', lw=1, label=f'Ambient dim D={D}')
ax.set_ylabel('ID estimate')
ax.set_title('MJO: ID estimators on real latents vs sanity controls', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/controls.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 4 — Stratified ID by ENSO Category (H1 vs H2 vs H3 Diagnostic)

Compute d̂ separately for {EN, Neutral, LN} subsets. The signal:
- **H1 / H3-without-ENSO**: within-category d̂ ≈ global d̂ (ENSO doesn't reduce the dimension when fixed)
- **H2 / H3-with-ENSO**: within-category d̂ ≈ global d̂ − 1 (fixing ENSO removes one state-space axis)

In [ ]:
id_by_enso = {}
for cat in ['El Nino', 'Neutral', 'La Nina']:
    mask = (enso == cat)
    n_cat = int(mask.sum())
    if n_cat < 200:
        id_by_enso[cat] = {'n': n_cat, 'tnn': float('nan'), 'mle': float('nan')}
        print(f'  {cat:9s}:  N = {n_cat:5d}  (too few — skipped)')
        continue
    z_cat = z[mask]
    k_cat = max(int(n_cat * 0.012), 3)
    tnn = float(skdim.id.TwoNN().fit(z_cat).dimension_)
    mle = float(skdim.id.MLE(K=k_cat).fit(z_cat).dimension_)
    id_by_enso[cat] = {'n': n_cat, 'tnn': tnn, 'mle': mle, 'k': k_cat}
    print(f'  {cat:9s}:  N = {n_cat:5d}  Two-NN = {tnn:.2f}  MLE(k={k_cat}) = {mle:.2f}')

global_tnn = id_tnn
global_mle = LB_mean

print('\nInterpretation (Two-NN-based):')
print(f'  Global d̂:        {global_tnn:.2f}')
for cat, r in id_by_enso.items():
    if not np.isnan(r['tnn']):
        delta = r['tnn'] - global_tnn
        if abs(delta) < 0.5:
            flag = '(≈ global → H1/H3-no-ENSO consistent)'
        elif -1.5 < delta < -0.5:
            flag = f'(Δ = {delta:+.2f} → H2/H3-with-ENSO consistent)'
        else:
            flag = f'(Δ = {delta:+.2f} → unusual)'
        print(f'  {cat:9s} d̂:     {r["tnn"]:.2f}  {flag}')

fig, ax = plt.subplots(figsize=(9, 5))
cats = ['El Nino', 'Neutral', 'La Nina']
x = np.arange(len(cats) + 1)
tnn_vals = [global_tnn] + [id_by_enso[c]['tnn'] for c in cats]
mle_vals = [global_mle] + [id_by_enso[c]['mle'] for c in cats]
ns       = [N]          + [id_by_enso[c]['n']   for c in cats]
labels   = ['Global'] + cats
w = 0.35
ax.bar(x - w/2, tnn_vals, width=w, color='#1f77b4', alpha=0.85, label='Two-NN')
ax.bar(x + w/2, mle_vals, width=w, color='steelblue', alpha=0.85, label='MLE (Levina-Bickel)')
for xi, (t, m, n) in enumerate(zip(tnn_vals, mle_vals, ns)):
    if not np.isnan(t): ax.text(xi - w/2, t + 0.10, f'{t:.2f}', ha='center', fontsize=10)
    if not np.isnan(m): ax.text(xi + w/2, m + 0.10, f'{m:.2f}', ha='center', fontsize=10)
    ax.text(xi, -0.5, f'N={n}', ha='center', fontsize=9, color='gray')
ax.axhline(global_tnn, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('ID estimate')
ax.set_title('MJO: ID stratified by ENSO category', fontweight='bold', fontsize=11)
ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.set_ylim(bottom=-1)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/id_by_enso.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 5 — 2-D PCA Visualization (BSISO Phase + ENSO Coloring)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=min(D, N - 1)).fit(z_centered)
z_pca = pca.transform(z_centered)
var_ratio = pca.explained_variance_ratio_
cum_var   = np.cumsum(var_ratio)
print(f'Variance explained by first PCs:')
for i in range(min(8, len(var_ratio))):
    print(f'  PC{i+1}: {var_ratio[i]*100:5.2f}%   (cum {cum_var[i]*100:5.2f}%)')

d_hat_idx = max(0, min(round(LB_mean) - 1, len(cum_var) - 1))
cum_at_dhat = float(cum_var[d_hat_idx])

fig = plt.figure(figsize=(18, 5))

ax = fig.add_subplot(1, 3, 1)
n_show = min(15, len(var_ratio))
ax.bar(range(1, n_show+1), var_ratio[:n_show]*100, color='steelblue', alpha=0.85)
ax.axvline(round(LB_mean), color='red', ls='--', lw=1.5, label=f'd̂ = {round(LB_mean)}')
ax.set_xlabel('PC index'); ax.set_ylabel('% variance explained')
ax.set_title(f'MJO scree (first {n_show} PCs)\nCum var at d̂ = {round(LB_mean)}: {cum_at_dhat*100:.1f}%',
             fontweight='bold', fontsize=11)
ax.legend(); ax.grid(alpha=0.3, axis='y')

ax = fig.add_subplot(1, 3, 2)
phase_colors = plt.cm.hsv(np.linspace(0, 1, 9))[:8]
for p in range(1, 9):
    m = phase == p
    ax.scatter(z_pca[m, 0], z_pca[m, 1], c=[phase_colors[p-1]], s=4, alpha=0.5, label=f'P{p}')
ax.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}%)')
ax.set_title('MJO latent colored by RMM phase\n(ring 1→...→8→1 if d̂=2)', fontweight='bold', fontsize=11)
ax.legend(fontsize=8, ncol=2, loc='best', markerscale=2); ax.grid(alpha=0.3); ax.set_aspect('equal')

ax = fig.add_subplot(1, 3, 3)
enso_cmap = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker = {'El Nino': '^', 'Neutral': 'o', 'La Nina': 's'}
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = enso == cat
    ax.scatter(z_pca[m, 0], z_pca[m, 1], c=enso_cmap[cat], marker=enso_marker[cat],
               s=4, alpha=0.45, label=f'{cat} (N={int(m.sum())})')
ax.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}%)')
ax.set_title('MJO latent colored by ENSO\n(H2/H3-with-ENSO: clear separation)', fontweight='bold', fontsize=11)
ax.legend(fontsize=9, loc='best', markerscale=2); ax.grid(alpha=0.3); ax.set_aspect('equal')

plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/pca_visualization.png', dpi=140, bbox_inches='tight'); plt.show()

n_pcs_90 = int(np.argmax(cum_var >= 0.90)) + 1 if (cum_var >= 0.90).any() else len(cum_var)
print(f'\nManifold compactness:')
print(f'  Cum var at d̂ = {round(LB_mean)} PCs:  {cum_at_dhat*100:.1f}%')
print(f'  PCs to reach 90%:               {n_pcs_90}')

## Cell 6 — Final Decision (H1/H2/H3) + Save

N-aware confidence threshold from Session 32 patch: trust d̂ if `noise > 3 × d̂` (≥ 3× headroom above the measurement).

In [ ]:
import time as _t

methods_close = (abs(id_tnn - LB_mean) <= 1.0) and (abs(id_lpca - LB_mean) <= 1.5)
lb_tight      = LB_std < 0.5
manifold_real = id_shuf_tnn > LB_mean + 0.5
d_hat         = max(1, round(LB_mean))
noise_calib   = id_noise_tnn > 3 * d_hat
noise_at_saturation = id_noise_tnn < D * 0.7

if not noise_calib:
    confidence = 'LOW'
    decision_text = (f'**Estimator headroom failed**: noise control returned {id_noise_tnn:.1f}, '
                     f'less than 3× the real d̂ = {d_hat}. Measurement too close to saturation ceiling.')
elif not manifold_real:
    confidence = 'LOW'
    decision_text = (f'**Manifold reality check failed**: shuffled-z d̂ = {id_shuf_tnn:.1f} '
                     f'is too close to real-data d̂ = {LB_mean:.1f}. Manifold structure not asserted.')
elif lb_tight and methods_close:
    confidence = 'HIGH'
    decision_text = (f'**HIGH-confidence ID estimate: d̂ = {d_hat}.** '
                     f'LB ({LB_mean:.2f}±{LB_std:.2f}), Two-NN ({id_tnn:.2f}), lPCA ({id_lpca:.2f}) agree. '
                     f'Shuffled control {id_shuf_tnn:.1f} ≫ {d_hat}. '
                     f'Noise ceiling {id_noise_tnn:.1f} (3× headroom). '
                     f'Proceed to nb23 with SIREN bottleneck = {d_hat}.')
elif lb_tight:
    confidence = 'MEDIUM'
    decision_text = (f'**MEDIUM-confidence ID estimate: d̂ = {d_hat}.** '
                     f'LB ({LB_mean:.2f}) tight; methods diverge: Two-NN ({id_tnn:.2f}), lPCA ({id_lpca:.2f}). '
                     f'Proceed to nb23 with bottleneck = {d_hat}, ablate {max(1, d_hat-1)} and {d_hat+1}.')
else:
    confidence = 'LOW'
    decision_text = f'**Low-confidence**: LB std {LB_std:.2f} > 0.5. Train multiple SIRENs and compare.'

# Three-hypothesis classification
if d_hat == 2:
    hypothesis = 'H1'
    interp = ('**H1 confirmed: MJO is intrinsically 2-D.** Wheeler-Hendon RMM (PC1, PC2) is '
              'dimension-sufficient. **Structural asymmetry with BSISO** (Session 32: d̂=4): '
              'MJO and BSISO differ fundamentally — MJO is a single equatorial coupled mode, '
              'BSISO has multiple monsoon-Rossby + amplitude + ENSO axes.')
elif d_hat == 3:
    hypothesis = 'H2'
    interp = ('**H2 confirmed: MJO needs 3 state variables.** The 3rd axis is likely RMM amplitude or '
              'ENSO modulation. Less than BSISO\'s 4 but still beyond the RMM convention.')
elif d_hat >= 4:
    hypothesis = 'H3'
    interp = (f'**H3 confirmed: MJO d̂ = {d_hat} ≥ 4, matching BSISO.** '
              f'Both intraseasonal modes have undercounted state spaces. '
              f'**Strongest possible result for the project**: the conventional 2-D intraseasonal '
              f'indices are systematically too compressed. nb23 will identify the {d_hat} physical correlates.')
else:
    hypothesis = 'unexpected'
    interp = f'd̂ = {d_hat} below 2 — unexpected. MJO has at least RMM PC1+PC2 → ID ≥ 2 expected.'

# Save JSON
result = {
    'date':                _t.strftime('%Y-%m-%d'),
    'pipeline':            'MJO NSV',
    'n_samples':           int(N),
    'ambient_dim':         int(D),
    'k_list':              [int(k) for k in k_list],
    'k_fractions':         k_fractions,
    'LB_by_k':             [float(v) for v in id_lb_by_k],
    'LB_mean':             float(LB_mean),
    'LB_std':              float(LB_std),
    'TwoNN':               float(id_tnn),
    'lPCA':                float(id_lpca),
    'd_hat':               int(d_hat),
    'hypothesis':          hypothesis,
    'confidence':          confidence,
    'controls': {
        'noise_TwoNN':     float(id_noise_tnn),
        'noise_MLE':       float(id_noise_mle),
        'shuffled_TwoNN':  float(id_shuf_tnn),
        'shuffled_MLE':    float(id_shuf_mle),
        'noise_calib_threshold': float(3 * d_hat),
        'noise_at_saturation':   bool(noise_at_saturation),
    },
    'id_by_enso':          id_by_enso,
    'pca_cumvar_at_dhat':  float(cum_at_dhat),
    'pca_pc1':             float(var_ratio[0]),
    'pca_pc2':             float(var_ratio[1]),
}
with open(f'{RESULTS_DIR}/intrinsic_dim.json', 'w') as f:
    json.dump(result, f, indent=2, default=float)

# Save markdown summary
summary_md = f"""# MJO NSV Stage 2 — Intrinsic Dimension Estimate

**Date:** {_t.strftime('%Y-%m-%d')}  
**Input:** {N} deduplicated 64-D latents from nb21 (MJO bp20-90, lag-10).

## Headline number

**d̂ = {d_hat}**  ({confidence} confidence)  →  **{hypothesis}**

| Method | Estimate |
|---|---|
| Levina-Bickel (k-sweep mean ± std) | **{LB_mean:.3f} ± {LB_std:.3f}** |
| Two-NN | {id_tnn:.3f} |
| local PCA | {id_lpca:.3f} |

## Sanity controls

| Control | Value | Verdict |
|---|---|---|
| Gaussian noise (Two-NN) | {id_noise_tnn:.2f} | {'✓' if noise_calib else '✗'} 3× headroom above d̂={d_hat} |
| Shuffled z (Two-NN) | {id_shuf_tnn:.2f} | {'✓' if manifold_real else '✗'} ≫ real d̂ |

## ENSO stratification

| Subset | N | Two-NN | MLE | Δ from global |
|---|---|---|---|---|
{chr(10).join(f'| {cat:9s} | {r["n"]:5d} | {r["tnn"]:.2f} | {r["mle"]:.2f} | {r["tnn"] - global_tnn:+.2f} |' for cat, r in id_by_enso.items() if not np.isnan(r['tnn']))}

## BSISO comparison (Session 32)

| | BSISO | MJO (this run) |
|---|:-:|:-:|
| d̂ | 4 | **{d_hat}** |
| Hypothesis | H3 | **{hypothesis}** |
| Improvement over persistence (Stage 1) | +18.9% | +63.4% |
| PC1 latent fraction | 85.8% | 76.0% |

## Decision

{decision_text}

## Scientific interpretation

{interp}
"""
with open(f'{RESULTS_DIR}/stage2_summary.md', 'w') as f:
    f.write(summary_md)

print('=' * 78)
print(f'  FINAL d̂ = {d_hat}   →   {hypothesis}   ({confidence} confidence)')
print('=' * 78)
print(decision_text)
print()
print('Scientific interpretation:')
print(interp)
print(f'\nSaved: {RESULTS_DIR}/intrinsic_dim.json')
print(f'Saved: {RESULTS_DIR}/stage2_summary.md')

---
## Done!

**Send back** for review:
1. Cell 6 final console block — `d̂`, hypothesis (H1/H2/H3), confidence.
2. `results/stage2_lag10/controls.png` — sanity that noise control is well above real d̂ and shuffled control is well above real d̂.
3. `results/stage2_lag10/pca_visualization.png` — RMM phase + ENSO scatter on the first two PCs. Look for ring structure (if H1) or ENSO cluster separation (if H2/H3-with-ENSO).
4. `results/stage2_lag10/id_by_enso.png` — within-cat d̂ vs global d̂ comparison.

**Once we have d̂, nb23** does Stage 3 + 4 + analysis: SIREN refine to d̂-dim bottleneck, dynamics MLP `v_t → v_{t+10}`, per-dim correlations with RMM phase/amplitude/ENSO/DOY, ENSO displacement z-score vs nb14 baseline (12.21) and nb16 RMM baseline (4.10).

---
*DDCS Project | jh9141@nyu.edu*